# 🚄 KTX/SRT 열차 예약 APK 빌더

이 노트북을 Google Colab에서 실행하면 Android APK를 빌드할 수 있습니다.

## 사용법
1. Google Colab에서 이 노트북을 엽니다
2. 프로젝트 소스코드를 업로드합니다
3. 각 셀을 순서대로 실행합니다
4. 빌드된 APK를 다운로드합니다

In [ ]:
# Step 1: 시스템 의존성 및 Buildozer 설치
!sudo apt-get update -qq
!sudo apt-get install -y -qq \
    python3-pip build-essential git ffmpeg \
    libsdl2-dev libsdl2-image-dev libsdl2-mixer-dev libsdl2-ttf-dev \
    libportmidi-dev libswscale-dev libavformat-dev libavcodec-dev \
    zlib1g-dev libgstreamer1.0-dev gstreamer1.0-plugins-base \
    libgstreamer-plugins-base1.0-dev libjpeg-dev libpng-dev libtiff-dev \
    libgl1-mesa-dev libgles2-mesa-dev autoconf automake libtool \
    pkg-config libffi-dev libssl-dev cmake unzip openjdk-17-jdk

!pip install -q buildozer cython==3.0.10 kivy==2.3.0

print('✅ 의존성 설치 완료!')

In [ ]:
# Step 2: 프로젝트 소스코드 업로드
# 옵션 A: GitHub에서 클론
# !git clone https://github.com/YOUR_USERNAME/Train_web.git /content/Train_web

# 옵션 B: ZIP 파일 업로드
from google.colab import files
print('프로젝트 ZIP 파일을 업로드하세요 (Train_web.zip)')
uploaded = files.upload()

import zipfile, os
for filename in uploaded.keys():
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print(f'✅ {filename} 압축 해제 완료!')

# 프로젝트 디렉토리 확인
!ls /content/Train_web/ 2>/dev/null || ls /content/

In [ ]:
# Step 3: 빌드 디렉토리 준비
import shutil, os

PROJECT_DIR = '/content/Train_web'  # 프로젝트 경로 (필요시 수정)
BUILD_DIR = '/content/apk_build'

# 기존 빌드 디렉토리 삭제
if os.path.exists(BUILD_DIR):
    shutil.rmtree(BUILD_DIR)
os.makedirs(BUILD_DIR)

# 파일 복사
shutil.copy2(f'{PROJECT_DIR}/mobile/main.py', f'{BUILD_DIR}/main.py')
shutil.copytree(f'{PROJECT_DIR}/app', f'{BUILD_DIR}/app')
shutil.copytree(f'{PROJECT_DIR}/korail2', f'{BUILD_DIR}/korail2')
shutil.copytree(f'{PROJECT_DIR}/SRT', f'{BUILD_DIR}/SRT')
shutil.copy2(f'{PROJECT_DIR}/mobile/buildozer.spec', f'{BUILD_DIR}/buildozer.spec')

# __pycache__ 정리
for root, dirs, _files in os.walk(BUILD_DIR):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))

print('✅ 빌드 디렉토리 준비 완료!')
!ls -la {BUILD_DIR}/

In [ ]:
# Step 4: APK 빌드 (30~60분 소요)
%cd /content/apk_build
!yes | buildozer android debug 2>&1 | tail -50

print('\n' + '='*50)
print('빌드 완료! APK 파일 확인 중...')
!find /content/apk_build -name '*.apk' -type f

In [ ]:
# Step 5: APK 다운로드
import glob
from google.colab import files

apk_files = glob.glob('/content/apk_build/**/*.apk', recursive=True)

if apk_files:
    apk_path = apk_files[0]
    # 이름 변경
    output_name = 'TrainReservation-v2.0.0.apk'
    shutil.copy2(apk_path, f'/content/{output_name}')
    
    size_mb = os.path.getsize(f'/content/{output_name}') / (1024*1024)
    print(f'✅ APK 빌드 성공!')
    print(f'📦 파일: {output_name}')
    print(f'📏 크기: {size_mb:.1f} MB')
    print()
    print('다운로드를 시작합니다...')
    files.download(f'/content/{output_name}')
else:
    print('❌ APK 파일을 찾을 수 없습니다.')
    print('빌드 로그를 확인해주세요.')